In [ ]:
from datetime import datetime
import os
import time
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

load_dotenv()

# -------------------------------------------------------------
# Configuration & Environment Variables
# -------------------------------------------------------------
search_url = os.getenv("PORTAL")
non_min_path = os.getenv("NonMin")
silver_u_dir = os.getenv("SILVERU")

if not non_min_path or not os.path.exists(non_min_path):
    raise FileNotFoundError(
        f"❌ NonMin environment path not found or invalid: {non_min_path}"
    )

if not silver_u_dir:
    raise ValueError("❌ SILVERU environment variable is not set.")

os.makedirs(silver_u_dir, exist_ok=True)

# -------------------------------------------------------------
# Initialize Browser
# -------------------------------------------------------------
chrome_options = webdriver.ChromeOptions()
chrome_options.page_load_strategy = "eager"
driver = webdriver.Chrome(options=chrome_options)
driver.get(search_url)
wait = WebDriverWait(driver, 10)

# -------------------------------------------------------------
# Load Excel File from NonMin
# -------------------------------------------------------------
print(f"📂 Loading file from NonMin path: {non_min_path}")
df_leads = pd.read_excel(non_min_path)

if (
    "Proposal No." not in df_leads.columns
    or "Proposal Status" not in df_leads.columns
):
    raise KeyError(
        "The Excel file must contain both 'Proposal No.' and 'Proposal Status' columns."
    )

print(
    f"Loaded {len(df_leads)} rows. Starting portal check for updated proposal statuses..."
)
main_window = driver.current_window_handle

updated_records = []

# -------------------------------------------------------------
# Search Loop & Status Verification
# -------------------------------------------------------------
for idx, row in df_leads.iterrows():
    proposal_no = str(row["Proposal No."]).strip()
    original_status = str(row["Proposal Status"]).strip()

    if (
        pd.isna(row["Proposal No."])
        or proposal_no == ""
        or proposal_no.lower() == "nan"
    ):
        continue

    print(f"\n[{idx + 1}/{len(df_leads)}] Checking Proposal: {proposal_no}")

    try:
        if (
            "trackYourProposal" not in driver.current_url
            or "proposal-details" in driver.current_url
        ):
            driver.get(search_url)

        proposal_input = wait.until(
            EC.visibility_of_element_located(
                (By.XPATH, "//input[@formcontrolname='proposalNumber']")
            )
        )
        proposal_input.clear()
        proposal_input.send_keys(proposal_no)

        search_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[@type='submit' and contains(.,'Search')]")
            )
        )
        driver.execute_script("arguments[0].click();", search_button)

        try:
            WebDriverWait(driver, 7).until(
                EC.text_to_be_present_in_element(
                    (
                        By.XPATH,
                        "//table[@id='excel-table']/tbody/tr[1]/td[2]",
                    ),
                    proposal_no,
                )
            )
        except TimeoutException:
            print(f"  ⚠️ No matching records found for: {proposal_no}. Skipping...")
            continue

        # Extract the Proposal Status precisely using the table header structure (7th column: td[7])
        try:
            status_cell = driver.find_element(
                By.XPATH,
                "//table[@id='excel-table']/tbody/tr[1]/td[7]",
            )
            portal_status = status_cell.text.strip()
        except NoSuchElementException:
            portal_status = "Unknown"

        print(
            f"  -> Original Status: '{original_status}' | Portal Status: '{portal_status}'"
        )

        # Compare status (case-insensitive and whitespace normalized check)
        if (
            portal_status
            and portal_status.lower() != "unknown"
            and portal_status.lower() != original_status.lower()
        ):
            print(
                f"  ✨ Status change detected! Updating status to: '{portal_status}'"
            )
            # Update dataframe row with new portal status
            df_leads.at[idx, "Proposal Status"] = portal_status
            updated_records.append(df_leads.loc[idx].copy())

        driver.get(search_url)
        wait.until(
            EC.visibility_of_element_located(
                (By.XPATH, "//input[@formcontrolname='proposalNumber']")
            )
        )

    except Exception as e:
        print(f" ❌ Processing fail for proposal {proposal_no}: {e}")
        all_windows = driver.window_handles
        if len(all_windows) > 1:
            for extra_w in all_windows[1:]:
                driver.switch_to.window(extra_w)
                driver.close()
        driver.switch_to.window(main_window)
        driver.get(search_url)
        continue

# Quit WebDriver
driver.quit()

# -------------------------------------------------------------
# Save Output File with Datestamp at SILVERU
# -------------------------------------------------------------
cdate = datetime.now().strftime("%Y%m%d")
output_file_name = f"NonMin_Updated_Status_{cdate}.xlsx"
output_file_path = os.path.join(silver_u_dir, output_file_name)

if updated_records:
    df_output = pd.DataFrame(updated_records)
    df_output.to_excel(output_file_path, index=False)
    print(
        f"\n🎉 Task complete! Found {len(updated_records)} changed records. Saved to: {output_file_path}"
    )
else:
    print(
        "\nℹ️ No status changes detected across any proposals. Saving full verified dataset snapshot."
    )
    df_leads.to_excel(output_file_path, index=False)
    print(f"📁 Saved snapshot to: {output_file_path}")

📂 Loading file from NonMin path: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\Non-Mineral_PotentialLeads.xlsx
Loaded 3931 rows. Starting portal check for updated proposal statuses...

[1/3931] Checking Proposal: IA/BR/IND1/568547/2026
  -> Original Status: 'INTIMATED' | Portal Status: 'INTIMATED'

[2/3931] Checking Proposal: IA/RJ/IND1/529995/2025
  -> Original Status: 'ToR Granted' | Portal Status: 'ToR Granted'

[3/3931] Checking Proposal: IA/GJ/IND1/462847/2024
  -> Original Status: 'INTIMATED' | Portal Status: 'INTIMATED'

[4/3931] Checking Proposal: IA/RJ/IND1/584596/2026
  -> Original Status: 'Under Processing with Authority' | Portal Status: 'Under Processing with Authority'

[5/3931] Checking Proposal: IA/CG/IND1/580033/2026
  -> Original Status: 'Under Examination' | Portal Status: 'Under Examination'

[6/3931] Checking Proposal: IA/RJ/IND1/565791/2026
  -> Original Status: 'Under Examination' | Portal Status: 'Under Examination'

[7/3931] Checking Proposal: 